# Feature Engineering



## 1. Load Dataset

We load the clean CSV file produced in the previous notebook.

In [ ]:
import pandas as pd

df = pd.read_csv("../database/data_cleaning/data_cleaning.csv", sep=",")

## 2. Separate Target Variable

We define what the model will predict:
- `y` → `Exited` (1 = left the bank, 0 = stayed)
- `x` → all remaining columns

In [2]:
x = df.drop(columns=["Exited"])
y = df["Exited"]

## 3. Create Dummy Variables

We convert categorical text columns into numeric format using `pd.get_dummies`.

- `drop_first=True` removes one category per variable to avoid multicollinearity.
- Columns transformed: `Geography`, `Gender`, `Card Type`.

In [3]:
categorical_cols = ["Geography", "Gender", "Card Type"]

x = pd.get_dummies(x, columns=categorical_cols, drop_first=True)

print(x.columns.tolist())



['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Point Earned', 'Geography_Germany', 'Geography_Spain', 'Gender_Male', 'Card Type_GOLD', 'Card Type_PLATINUM', 'Card Type_SILVER']


## 4. Two Versions of X

We build two feature matrices to evaluate both scenarios discussed in the data cleaning stage:

- `x_full` → all features including `Complain`
- `x_no_complain` → all features excluding `Complain` (avoids potential data leakage)

In [4]:
x_full = x.copy()

x_no_complain = x.drop(columns=["Complain"])


## 5. Train / Test Split

We split both versions into train (80%) and test (20%).

- `stratify=y` ensures both sets maintain the same proportion of churners (20/80).
- `random_state=123` makes the split reproducible.

In [5]:
from sklearn.model_selection import train_test_split

x_full_train, x_full_test, y_train, y_test = train_test_split(x_full, y, test_size=0.2, random_state= 123, stratify=y )
x_no_comp_train, x_no_comp_test, _, _ = train_test_split( x_no_complain, y, test_size=0.2, random_state=123, stratify=y)

print(f"x_full_train:       {x_full_train.shape}")
print(f"x_full_test:        {x_full_test.shape}")
print(f"x_no_comp_train:    {x_no_comp_train.shape}")
print(f"x_no_comp_test:     {x_no_comp_test.shape}")
print(f"y_train:            {y_train.shape}")
print(f"y_test:             {y_test.shape}")

x_full_train:       (8000, 17)
x_full_test:        (2000, 17)
x_no_comp_train:    (8000, 16)
x_no_comp_test:     (2000, 16)
y_train:            (8000,)
y_test:             (2000,)


## 6. Scale Numeric Features

Logistic Regression is sensitive to scale. We apply `StandardScaler` to bring all continuous numeric variables to mean = 0 and std = 1.

- The scaler is **fitted only on train** to avoid data leakage into the test set.
- Binary and dummy columns are not scaled.
- Columns scaled: `CreditScore`, `Age`, `Tenure`, `Balance`, `EstimatedSalary`, `Satisfaction Score`, `Point Earned`.

In [6]:
from sklearn.preprocessing import StandardScaler
cols_to_scale = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Satisfaction Score", "Point Earned"]

scaler = StandardScaler()

x_full_train[cols_to_scale] = scaler.fit_transform(x_full_train[cols_to_scale])
x_full_test[cols_to_scale]  = scaler.transform(x_full_test[cols_to_scale])

x_no_comp_train[cols_to_scale] = scaler.fit_transform(x_no_comp_train[cols_to_scale])
x_no_comp_test[cols_to_scale]  = scaler.transform(x_no_comp_test[cols_to_scale])

print(x_full_train[cols_to_scale].describe().round(2))

       CreditScore      Age   Tenure  Balance  EstimatedSalary  \
count      8000.00  8000.00  8000.00  8000.00          8000.00   
mean         -0.00     0.00     0.00    -0.00             0.00   
std           1.00     1.00     1.00     1.00             1.00   
min          -3.10    -2.02    -1.74    -1.22            -1.75   
25%          -0.69    -0.67    -0.70    -1.22            -0.85   
50%           0.01    -0.18    -0.01     0.33             0.00   
75%           0.70     0.49     0.69     0.82             0.86   
max           2.07     5.13     1.72     2.80             1.74   

       Satisfaction Score  Point Earned  
count             8000.00       8000.00  
mean                 0.00          0.00  
std                  1.00          1.00  
min                 -1.43         -1.96  
25%                 -0.72         -0.88  
50%                 -0.01         -0.01  
75%                  0.70          0.86  
max                  1.41          1.74  


## 7. Save Datasets

We export all six datasets as CSV files so the modeling notebook can load them directly.

In [7]:
path = "C:/Users/Pablito/Desktop/2026.1_Python_FE/database/data_cleaning/"

x_full_train.to_csv(path + "x_full_train.csv", index=False)
x_full_test.to_csv(path + "x_full_test.csv", index=False)
x_no_comp_train.to_csv(path + "x_no_comp_train.csv", index=False)
x_no_comp_test.to_csv(path + "x_no_comp_test.csv", index=False)
y_train.to_csv(path + "y_train.csv", index=False)
y_test.to_csv(path + "y_test.csv", index=False)
